# Which dates?

Each of the three methods below work independently. Moving forward none are really necessary, as we can gather weather information directly after cleaning the hive files; no need to read the files just written. Still, you can take these as an exercise in file reading and working with parquet files.

First, we'll name the folder containing our files.

In [1]:
root_dir = '/Users/cmontefusco/Documents/DSAI2/9. Azure/Data/'
source = root_dir+'silver/'

## 1. pandas

Most straightforward and familiar to us, we can read the files with pandas and use `.min()` and `.max()` to find the least and greatest values in the `"timestamp"` comlumns.

In [4]:
import pandas as pd

flow = pd.read_parquet(source+"flow/schwartau__2025-08-13T12h35m42s.parquet")
humidity = pd.read_parquet(source+"humidity/schwartau__2025-08-13T12h35m42s.parquet")
temperature = pd.read_parquet(source+"temperature/schwartau__2025-08-13T12h35m42s.parquet")
weight = pd.read_parquet(source+"weight/schwartau__2025-08-13T12h35m42s.parquet")

start_date = min(flow["timestamp"].min(), humidity["timestamp"].min(), temperature["timestamp"].min(), weight["timestamp"].min())
end_date = max(flow["timestamp"].max(), humidity["timestamp"].max(), temperature["timestamp"].max(), weight["timestamp"].max())

start_date, end_date

(Timestamp('2017-01-01 12:00:00+0000', tz='UTC'),
 Timestamp('2019-05-31 12:15:00+0000', tz='UTC'))

## 2. Parquet metadata

Parquet files write data into row-groups, and record metadata about those row-groups. We can explore the metadata of our parquet files to locate the column containing dates, then collect the statistics of that column in each row-group. Finally, we select the minimum and maximum values of all collected dates.

In [7]:
import pyarrow.parquet as pq

files = [
    source+"flow/schwartau__2025-08-13T12h35m42s.parquet",
    source+"humidity/schwartau__2025-08-13T12h35m42s.parquet",
    source+"temperature/schwartau__2025-08-13T12h35m42s.parquet",
    source+"weight/schwartau__2025-08-13T12h35m42s.parquet"
]

dates = []
for file in files:
    par = pq.ParquetFile(file)
    for rg in range(par.num_row_groups):
        stats = par.metadata.row_group(rg).column(0).statistics
        # print(stats)
        dates.extend([stats.min, stats.max])

start_date = min(dates)
end_date = max(dates)

start_date, end_date

(Timestamp('2017-01-01 12:00:00+0000', tz='UTC'),
 Timestamp('2019-05-31 12:15:00+0000', tz='UTC'))

## 3. OS and parquet metadata

The first two methods involved directly naming our files. This last one adds one extra challenge by using `os.listdir()` to find the names of our files before collecting their metadata.

In [10]:
import os
import pyarrow.parquet as pq

dates = []
for folder in os.listdir(source):
    # print(folder)
    if folder.startswith("."): # ignore 'hidden' folders
        continue
    for file in os.listdir(source+folder):
        # print(file)
        if file.endswith(".parquet"):
            par = pq.ParquetFile(source+folder+"/"+file)
            for rg in range(par.num_row_groups):
                stats = par.metadata.row_group(rg).column(0).statistics
                dates.extend([stats.min, stats.max])

start_date = min(dates)
end_date = max(dates)

start_date, end_date

(Timestamp('2017-01-01 12:00:00+0000', tz='UTC'),
 Timestamp('2019-05-31 12:15:00+0000', tz='UTC'))

# Weather

## 1. Coordinates

Eventually, we'll have two locations, so let's try to set ourselves up for the future. Coordinates can be organized into a dictionary by location.

In [15]:
coords = {
    "schwartau": {"lat": 53.919444, "lon": 10.6975}
}

We'll want to be able to combine the inner dictionary with more parameters to fill out our API call. How to do so?

Dictionaries cannot be "added" with the plus operator (`+`) in python, and `.update()` changes a dictionary in-place, which won't be useful. A little searching, however, reveals that the or operator (`|`) will allow us to "merge" two dictionaries. See the example below.

In [18]:
d1 = {"breakfast": "eggs"}
d2 = {"lunch": "toast"}
d1|d2

{'breakfast': 'eggs', 'lunch': 'toast'}

Here's how it would look with our dictionary of coordinates.

In [21]:
coords["schwartau"] | {"date": start_date}

{'lat': 53.919444,
 'lon': 10.6975,
 'date': Timestamp('2017-01-01 12:00:00+0000', tz='UTC')}

## 2. One day

We start easy, and gather weather data regarding just one day.

In [25]:
import requests
import pandas as pd

url = "https://api.brightsky.dev/weather"
headers = {"Accept": "application/json"}
params = coords["schwartau"] | {"date": start_date}

response = requests.get(url, headers=headers, params=params)

print(response.status_code)

200


`.status_code == 200`, so we can continue with processing data.

In [28]:
# transform to JSON (nested dictionary)
weather = response.json()

In [30]:
# write raw data to file
import json

weather_sink = root_dir+"bronze/archive/weather/"

with open (weather_sink+f"schwartau__{start_date.strftime('%Y-%m-%d')}.json", "w") as f:
    json.dump(weather, f, indent=4)

In [32]:
# check what keys are in the data
weather.keys()

dict_keys(['weather', 'sources'])

### 2.1 "weather" key

In [35]:
# what does the first value look like?
weather["weather"]

[{'timestamp': '2017-01-01T12:00:00+00:00',
  'source_id': 286541,
  'precipitation': 0.0,
  'pressure_msl': 1013.8,
  'sunshine': 0.0,
  'temperature': 5.2,
  'wind_direction': 230,
  'wind_speed': 10.8,
  'cloud_cover': 100,
  'dew_point': 3.5,
  'relative_humidity': 89,
  'visibility': 5450,
  'wind_gust_direction': 240,
  'wind_gust_speed': 26.6,
  'condition': 'dry',
  'precipitation_probability': None,
  'precipitation_probability_6h': None,
  'solar': 0.036,
  'fallback_source_ids': {'wind_speed': 7102,
   'wind_gust_speed': 7102,
   'solar': 7102,
   'pressure_msl': 7102,
   'dew_point': 7102,
   'temperature': 7102,
   'visibility': 7102,
   'cloud_cover': 7102,
   'wind_direction': 7102,
   'sunshine': 7102,
   'wind_gust_direction': 7102,
   'relative_humidity': 7102},
  'icon': 'cloudy'},
 {'timestamp': '2017-01-01T13:00:00+00:00',
  'source_id': 286541,
  'precipitation': 0.0,
  'pressure_msl': 1012.0,
  'sunshine': 0.0,
  'temperature': 5.8,
  'wind_direction': 240,
  'wi

In [37]:
# can we convert directly to DataFrame?
pd.json_normalize(weather["weather"])

,timestamp,source_id,precipitation,pressure_msl,sunshine,temperature,wind_direction,wind_speed,cloud_cover,dew_point,...,fallback_source_ids.solar,fallback_source_ids.pressure_msl,fallback_source_ids.dew_point,fallback_source_ids.temperature,fallback_source_ids.visibility,fallback_source_ids.cloud_cover,fallback_source_ids.wind_direction,fallback_source_ids.sunshine,fallback_source_ids.wind_gust_direction,fallback_source_ids.relative_humidity
0,2017-01-01T12:00:00+00:00,286541,0.0,1013.8,0.0,5.2,230.0,10.8,100.0,3.5,...,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0
1,2017-01-01T13:00:00+00:00,286541,0.0,1012.0,0.0,5.8,240.0,21.6,100.0,4.1,...,NaN,7174.0,7174.0,7174.0,7174.0,7174.0,7174.0,7174.0,7174.0,7174.0
2,2017-01-01T14:00:00+00:00,286541,0.0,1012.5,0.0,5.3,230.0,26.6,87.0,3.1,...,96577.0,96577.0,96577.0,96577.0,96577.0,96577.0,96577.0,96577.0,96577.0,96577.0
3,2017-01-01T15:00:00+00:00,286541,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2017-01-01T16:00:00+00:00,286541,0.0,1013.2,0.0,3.8,240.0,11.9,100.0,2.3,...,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0
5,2017-01-01T17:00:00+00:00,286541,0.0,1013.4,0.0,3.6,240.0,10.1,87.0,2.1,...,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0
6,2017-01-01T18:00:00+00:00,286541,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2017-01-01T19:00:00+00:00,286541,0.3,1013.3,0.0,3.6,250.0,15.5,87.0,2.1,...,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0
8,2017-01-01T20:00:00+00:00,286541,0.3,1013.5,0.0,2.9,250.0,15.5,87.0,2.2,...,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0
9,2017-01-01T21:00:00+00:00,286541,0.3,1013.8,NaN,2.4,260.0,15.1,87.0,1.6,...,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,7102.0,NaN,7102.0,7102.0


Converting directly to DataFrame works, but the column "fallback_source_ids" contains a dictionary. This isn't ideal. We'll check in the next round if this column adds any valuable information.

### 2.2 "sources" key

In [41]:
weather["sources"]

[{'id': 286541,
  'dwd_station_id': '04602',
  'observation_type': 'historical',
  'lat': 53.9385,
  'lon': 10.6983,
  'height': 27.0,
  'station_name': 'Schwartau,Bad -Groß Parin',
  'wmo_station_id': 'A791',
  'first_record': '2010-01-01T00:00:00+00:00',
  'last_record': '2025-02-25T23:00:00+00:00',
  'distance': 2122.0},
 {'id': 7102,
  'dwd_station_id': '03086',
  'observation_type': 'historical',
  'lat': 53.8025,
  'lon': 10.6989,
  'height': 14.74,
  'station_name': 'Lübeck-Blankensee',
  'wmo_station_id': '10156',
  'first_record': '2010-01-01T00:00:00+00:00',
  'last_record': '2025-08-16T23:00:00+00:00',
  'distance': 13018.0},
 {'id': 7174,
  'dwd_station_id': '03897',
  'observation_type': 'historical',
  'lat': 54.0893,
  'lon': 10.8773,
  'height': 2.18,
  'station_name': 'Pelzerhaken',
  'wmo_station_id': '10152',
  'first_record': '2010-01-01T00:00:00+00:00',
  'last_record': '2025-08-16T23:00:00+00:00',
  'distance': 22269.0},
 {'id': 96577,
  'dwd_station_id': '00596',

In [43]:
pd.DataFrame(weather["sources"])

,id,dwd_station_id,observation_type,lat,lon,height,station_name,wmo_station_id,first_record,last_record,distance
0,286541,04602,historical,53.9385,10.6983,27.00,"Schwartau,Bad -Groß Parin",A791,2010-01-01T00:00:00+00:00,2025-02-25T23:00:00+00:00,2122.0
1,7102,03086,historical,53.8025,10.6989,14.74,Lübeck-Blankensee,10156,2010-01-01T00:00:00+00:00,2025-08-16T23:00:00+00:00,13018.0
2,7174,03897,historical,54.0893,10.8773,2.18,Pelzerhaken,10152,2010-01-01T00:00:00+00:00,2025-08-16T23:00:00+00:00,22269.0
3,96577,00596,historical,54.0027,11.1908,15.00,Boltenhagen,10161,2010-01-01T00:00:00+00:00,2025-08-16T23:00:00+00:00,33611.0
4,7338,06163,historical,54.1654,10.3519,27.03,Dörnick,10150,2010-01-01T00:00:00+00:00,2025-08-16T23:00:00+00:00,35496.0


This is a much more manageable DataFrame. Will the two need to be combined?

Also, look at some of those station distances. Do we need to filter any of these out?

### 2.3 A unified DataFrame

In [47]:
weather_df = pd.merge(pd.json_normalize(weather["weather"]), pd.DataFrame(weather["sources"]), 
                      left_on="source_id", right_on="id", how="left")

weather_df = weather_df.drop(["source_id", "visibility", "condition", "icon", "precipitation_probability", "precipitation_probability_6h", "id", "observation_type", "first_record", "last_record"], axis=1)
weather_df = weather_df.rename({"lat": "station_lat", "lon": "station_lon", "height": "station_elevation"}, axis=1)
weather_df

,timestamp,precipitation,pressure_msl,sunshine,temperature,wind_direction,wind_speed,cloud_cover,dew_point,relative_humidity,...,fallback_source_ids.sunshine,fallback_source_ids.wind_gust_direction,fallback_source_ids.relative_humidity,dwd_station_id,station_lat,station_lon,station_elevation,station_name,wmo_station_id,distance
0,2017-01-01T12:00:00+00:00,0.0,1013.8,0.0,5.2,230.0,10.8,100.0,3.5,89.0,...,7102.0,7102.0,7102.0,04602,53.9385,10.6983,27.0,"Schwartau,Bad -Groß Parin",A791,2122.0
1,2017-01-01T13:00:00+00:00,0.0,1012.0,0.0,5.8,240.0,21.6,100.0,4.1,89.0,...,7174.0,7174.0,7174.0,04602,53.9385,10.6983,27.0,"Schwartau,Bad -Groß Parin",A791,2122.0
2,2017-01-01T14:00:00+00:00,0.0,1012.5,0.0,5.3,230.0,26.6,87.0,3.1,86.0,...,96577.0,96577.0,96577.0,04602,53.9385,10.6983,27.0,"Schwartau,Bad -Groß Parin",A791,2122.0
3,2017-01-01T15:00:00+00:00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,04602,53.9385,10.6983,27.0,"Schwartau,Bad -Groß Parin",A791,2122.0
4,2017-01-01T16:00:00+00:00,0.0,1013.2,0.0,3.8,240.0,11.9,100.0,2.3,90.0,...,7102.0,7102.0,7102.0,04602,53.9385,10.6983,27.0,"Schwartau,Bad -Groß Parin",A791,2122.0
5,2017-01-01T17:00:00+00:00,0.0,1013.4,0.0,3.6,240.0,10.1,87.0,2.1,90.0,...,7102.0,7102.0,7102.0,04602,53.9385,10.6983,27.0,"Schwartau,Bad -Groß Parin",A791,2122.0
6,2017-01-01T18:00:00+00:00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,04602,53.9385,10.6983,27.0,"Schwartau,Bad -Groß Parin",A791,2122.0
7,2017-01-01T19:00:00+00:00,0.3,1013.3,0.0,3.6,250.0,15.5,87.0,2.1,90.0,...,7102.0,7102.0,7102.0,04602,53.9385,10.6983,27.0,"Schwartau,Bad -Groß Parin",A791,2122.0
8,2017-01-01T20:00:00+00:00,0.3,1013.5,0.0,2.9,250.0,15.5,87.0,2.2,95.0,...,7102.0,7102.0,7102.0,04602,53.9385,10.6983,27.0,"Schwartau,Bad -Groß Parin",A791,2122.0
9,2017-01-01T21:00:00+00:00,0.3,1013.8,NaN,2.4,260.0,15.1,87.0,1.6,95.0,...,NaN,7102.0,7102.0,04602,53.9385,10.6983,27.0,"Schwartau,Bad -Groß Parin",A791,2122.0


In [49]:
weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])
weather_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 32 columns):
 #   Column                                   Non-Null Count  Dtype              
---  ------                                   --------------  -----              
 0   timestamp                                25 non-null     datetime64[ns, UTC]
 1   precipitation                            25 non-null     float64            
 2   pressure_msl                             17 non-null     float64            
 3   sunshine                                 16 non-null     float64            
 4   temperature                              17 non-null     float64            
 5   wind_direction                           17 non-null     float64            
 6   wind_speed                               17 non-null     float64            
 7   cloud_cover                              17 non-null     float64            
 8   dew_point                                17 non-null     float64        